In [1]:
from pathlib import Path

import pandas as pd

PROCESSED_DATA_DIR = Path("../data/processed")

customers = pd.read_csv(
    PROCESSED_DATA_DIR / "customers.csv",
    parse_dates=["signup_date"],
)

products = pd.read_csv(
    PROCESSED_DATA_DIR / "products.csv",
)

orders = pd.read_csv(
    PROCESSED_DATA_DIR / "orders.csv",
    parse_dates=["order_date", "delivery_date"],
)

order_items = pd.read_csv(
    PROCESSED_DATA_DIR / "order_items.csv",
)

payments = pd.read_csv(
    PROCESSED_DATA_DIR / "payments.csv",
)

reviews = pd.read_csv(
    PROCESSED_DATA_DIR / "reviews.csv",
    parse_dates=["review_date"],
)

In [2]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
}

In [3]:
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

customers: (1000, 5)
products: (100, 6)
orders: (3200, 7)
order_items: (5376, 7)
payments: (3200, 5)
reviews: (2019, 6)


In [6]:
delivered_orders = orders.loc[
    orders["order_status"].eq("delivered")
].copy()

print("Total delivered orders:", delivered_orders["order_id"].nunique())

Total delivered orders: 2627


In [13]:
order_items.columns


Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price_mad',
       'discount_rate', 'line_total_mad'],
      dtype='str')

In [14]:

delivered_orders.columns

Index(['order_id', 'customer_id', 'order_date', 'order_status',
       'delivery_date', 'shipping_city', 'shipping_fee_mad'],
      dtype='str')

In [16]:
delivered_sales = order_items.merge(
    delivered_orders[
        ["order_id", "order_date", "shipping_city"]
    ],
    on="order_id",
    how="inner",
    validate="many_to_one",
)

delivered_sales.head()

,order_item_id,order_id,product_id,quantity,unit_price_mad,discount_rate,line_total_mad,order_date,shipping_city
0,I000001,O00001,P099,3,243.24,0.10,656.75,2024-04-09,tangier
1,I000002,O00001,P014,1,3643.31,0.00,3643.31,2024-04-09,tangier
2,I000003,O00001,P071,2,2497.82,0.05,4745.86,2024-04-09,tangier
3,I000004,O00002,P075,1,2409.28,0.05,2288.82,2025-11-19,marrakech
4,I000005,O00002,P059,3,376.57,0.05,1073.22,2025-11-19,marrakech


In [17]:
total_revenue = delivered_sales["line_total_mad"].sum()

total_orders = delivered_orders["order_id"].nunique()

total_units_sold = delivered_sales["quantity"].sum()

average_order_value = (
    total_revenue / total_orders
    if total_orders > 0
    else 0
)

sales_kpis = pd.DataFrame([
    {
        "metric": "Total Revenue",
        "value": total_revenue,
        "unit": "MAD",
    },
    {
        "metric": "Delivered Orders",
        "value": total_orders,
        "unit": "Orders",
    },
    {
        "metric": "Units Sold",
        "value": total_units_sold,
        "unit": "Units",
    },
    {
        "metric": "Average Order Value",
        "value": average_order_value,
        "unit": "MAD",
    },
])

sales_kpis

,metric,value,unit
0,Total Revenue,1.278431e+07,MAD
1,Delivered Orders,2.627000e+03,Orders
2,Units Sold,7.040000e+03,Units
3,Average Order Value,4.866506e+03,MAD


In [18]:
delivered_sales["order_month"] = (
    delivered_sales["order_date"]
    .dt.to_period("M")
)

In [19]:
monthly_sales = (
    delivered_sales
    .groupby("order_month")
    .agg(
        revenue_mad=("line_total_mad", "sum"),
        orders=("order_id", "nunique"),
        units_sold=("quantity", "sum"),
    )
    .reset_index()
)

monthly_sales["average_order_value"] = (
    monthly_sales["revenue_mad"]
    / monthly_sales["orders"]
)

monthly_sales

,order_month,revenue_mad,orders,units_sold,average_order_value
0,2024-01,552673.23,113,305,4890.913540
1,2024-02,455962.56,91,254,5010.577582
2,2024-03,566231.42,105,313,5392.680190
3,2024-04,531824.80,110,285,4834.770909
4,2024-05,500338.90,105,288,4765.132381
5,2024-06,618872.42,118,337,5244.681525
6,2024-07,503852.10,112,290,4498.679464
7,2024-08,634402.91,118,346,5376.295847
8,2024-09,430956.84,102,254,4225.067059
9,2024-10,468310.34,105,272,4460.098476


In [20]:
best_month = monthly_sales.loc[
    monthly_sales["revenue_mad"].idxmax()
]

best_month

order_month                2024-11
revenue_mad              649240.47
orders                         127
units_sold                     351
average_order_value    5112.129685
Name: 10, dtype: object

In [ ]:
monthly_sales["revenue_growth_pct"] = (
    monthly_sales["revenue_mad"]
    .pct_change()
    .mul(100)
    .round(2)
)

monthly_sales

In [ ]:
# Inspect product columns before joining
print(products.columns.tolist())
print(order_items.columns.tolist())

['product_id', 'product_name', 'category', 'unit_price_mad', 'cost_price_mad', 'stock_status']
['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price_mad', 'discount_rate', 'line_total_mad']


In [26]:
product_sales = delivered_sales.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

product_sales.sample()

,order_item_id,order_id,product_id,quantity,unit_price_mad_x,discount_rate,line_total_mad,order_date,shipping_city,order_month,product_name,category,unit_price_mad_y,cost_price_mad,stock_status,_merge
1798,I002232,O01324,P094,1,245.02,0.05,232.77,2024-04-04,rabat,2024-04,Charger 2,accessories,245.02,137.64,In Stock,both


In [27]:
product_sales["_merge"].value_counts()

_merge
both          4395
left_only        0
right_only       0
Name: count, dtype: int64

In [ ]:
#Analyze product performance
product_performance = (
    product_sales
    .groupby(["product_id", "category"])
    .agg(
        revenue_mad=("line_total_mad", "sum"),
        units_sold=("quantity", "sum"),
        orders=("order_id", "nunique"),
    )
    .reset_index()
)

product_performance["average_selling_price_mad"] = (
    product_performance["revenue_mad"]
    / product_performance["units_sold"]
)

product_performance = product_performance.sort_values(
    "revenue_mad",
    ascending=False,
)

product_performance.head(10)

,product_id,category,revenue_mad,units_sold,orders,average_selling_price_mad
24,P025,computers,543941.51,91,56,5977.379231
33,P034,computers,384071.82,59,39,6509.691864
39,P040,computers,365124.35,71,44,5142.596479
35,P036,computers,341614.81,68,44,5023.747206
22,P023,computers,338732.24,81,51,4181.879506
16,P017,electronics,331802.20,99,57,3351.537374
21,P022,computers,316167.99,58,34,5451.172241
4,P005,electronics,296288.20,93,55,3185.894624
26,P027,computers,292473.43,69,43,4238.745362
2,P003,electronics,290334.20,79,52,3675.116456


In [29]:
# Analyze category performance
category_performance = (
    product_sales
    .groupby("category")
    .agg(
        revenue_mad=("line_total_mad", "sum"),
        units_sold=("quantity", "sum"),
        orders=("order_id", "nunique"),
        unique_products=("product_id", "nunique"),
    )
    .reset_index()
    .sort_values("revenue_mad", ascending=False)
)

category_performance

,category,revenue_mad,units_sold,orders,unique_products
1,computers,4942738.85,1355,764,20
2,electronics,3927222.06,1407,796,20
3,gaming,2739090.95,1484,833,20
4,home,973437.25,1487,821,20
0,accessories,201821.59,1307,752,20


In [30]:
total_revenue = category_performance["revenue_mad"].sum()

category_performance["revenue_share_pct"] = (
    category_performance["revenue_mad"]
    / total_revenue
    * 100
).round(2)

category_performance

,category,revenue_mad,units_sold,orders,unique_products,revenue_share_pct
1,computers,4942738.85,1355,764,20,38.66
2,electronics,3927222.06,1407,796,20,30.72
3,gaming,2739090.95,1484,833,20,21.43
4,home,973437.25,1487,821,20,7.61
0,accessories,201821.59,1307,752,20,1.58


In [31]:
assert "cost_price_mad" in product_sales.columns
assert "quantity" in product_sales.columns

In [32]:
product_sales["estimated_cost_mad"] = (
    product_sales["quantity"]
    * product_sales["cost_price_mad"]
)

product_sales["estimated_gross_profit_mad"] = (
    product_sales["line_total_mad"]
    - product_sales["estimated_cost_mad"]
)

In [33]:
product_profitability = (
    product_sales
    .groupby(["product_id", "category"])
    .agg(
        revenue_mad=("line_total_mad", "sum"),
        cost_mad=("estimated_cost_mad", "sum"),
        gross_profit_mad=(
            "estimated_gross_profit_mad",
            "sum",
        ),
        units_sold=("quantity", "sum"),
    )
    .reset_index()
)

product_profitability["gross_margin_pct"] = (
    product_profitability["gross_profit_mad"]
    / product_profitability["revenue_mad"]
    * 100
).round(2)

product_profitability = product_profitability.sort_values(
    "gross_profit_mad",
    ascending=False,
)

product_profitability.head(10)

,product_id,category,revenue_mad,cost_mad,gross_profit_mad,units_sold,gross_margin_pct
24,P025,computers,543941.51,356229.51,187712.00,91,34.51
16,P017,electronics,331802.20,215110.17,116692.03,99,35.17
22,P023,computers,338732.24,234830.34,103901.90,81,30.67
39,P040,computers,365124.35,263452.60,101671.75,71,27.85
14,P015,electronics,289321.61,198602.40,90719.21,83,31.36
32,P033,computers,229886.25,139635.36,90250.89,72,39.26
35,P036,computers,341614.81,253368.00,88246.81,68,25.83
1,P002,electronics,219200.43,136302.39,82898.04,63,37.82
79,P080,gaming,202210.08,120666.18,81543.90,79,40.33
23,P024,computers,233754.91,154009.02,79745.89,66,34.12


In [34]:
products_with_sales = product_performance[
    ["product_id", "units_sold", "revenue_mad"]
]

product_sales_coverage = products.merge(
    products_with_sales,
    on="product_id",
    how="left",
    validate="one_to_one",
)

product_sales_coverage[
    ["units_sold", "revenue_mad"]
] = product_sales_coverage[
    ["units_sold", "revenue_mad"]
].fillna(0)

products_without_sales = product_sales_coverage.loc[
    product_sales_coverage["units_sold"].eq(0)
]

products_without_sales.head()

,product_id,product_name,category,unit_price_mad,cost_price_mad,stock_status,units_sold,revenue_mad


## 4. Geographic Performance Analysis

In [36]:
city_performance = (
    delivered_sales
    .groupby("shipping_city")
    .agg(
        revenue_mad=("line_total_mad", "sum"),
        orders=("order_id", "nunique"),
        units_sold=("quantity", "sum"),
    )
    .reset_index()
)

city_performance["average_order_value_mad"] = (
    city_performance["revenue_mad"]
    / city_performance["orders"]
)

city_performance = city_performance.sort_values(
    "revenue_mad",
    ascending=False,
).reset_index(drop=True)

city_performance

,shipping_city,revenue_mad,orders,units_sold,average_order_value_mad
0,casablanca,2983973.57,606,1620,4924.048795
1,marrakech,1902172.73,382,1041,4979.509764
2,rabat,1340689.52,284,767,4720.737746
3,tangier,1160065.07,252,666,4603.432817
4,fes,1126717.93,232,639,4856.542802
5,agadir,986092.51,188,521,5245.172926
6,meknes,724358.53,148,386,4894.314392
7,tetouan,659212.47,123,341,5359.450976
8,kenitra,613984.52,150,359,4093.230133
9,oujda,580665.95,129,324,4501.286434


In [37]:
total_revenue = city_performance["revenue_mad"].sum()

city_performance["revenue_share_pct"] = (
    city_performance["revenue_mad"]
    / total_revenue
    * 100
).round(2)

city_performance

,shipping_city,revenue_mad,orders,units_sold,average_order_value_mad,revenue_share_pct
0,casablanca,2983973.57,606,1620,4924.048795,23.34
1,marrakech,1902172.73,382,1041,4979.509764,14.88
2,rabat,1340689.52,284,767,4720.737746,10.49
3,tangier,1160065.07,252,666,4603.432817,9.07
4,fes,1126717.93,232,639,4856.542802,8.81
5,agadir,986092.51,188,521,5245.172926,7.71
6,meknes,724358.53,148,386,4894.314392,5.67
7,tetouan,659212.47,123,341,5359.450976,5.16
8,kenitra,613984.52,150,359,4093.230133,4.80
9,oujda,580665.95,129,324,4501.286434,4.54


In [40]:
city_customers = (
    delivered_orders
    .groupby("shipping_city")
    .agg(
        unique_customers=("customer_id", "nunique"),
    )
    .reset_index()
)

city_performance = city_performance.merge(
    city_customers,
    on="shipping_city",
    how="left",
    validate="one_to_one",
)

city_performance

,shipping_city,revenue_mad,orders,units_sold,average_order_value_mad,revenue_share_pct,unique_customers
0,casablanca,2983973.57,606,1620,4924.048795,23.34,221
1,marrakech,1902172.73,382,1041,4979.509764,14.88,133
2,rabat,1340689.52,284,767,4720.737746,10.49,106
3,tangier,1160065.07,252,666,4603.432817,9.07,87
4,fes,1126717.93,232,639,4856.542802,8.81,78
5,agadir,986092.51,188,521,5245.172926,7.71,71
6,meknes,724358.53,148,386,4894.314392,5.67,50
7,tetouan,659212.47,123,341,5359.450976,5.16,44
8,kenitra,613984.52,150,359,4093.230133,4.80,55
9,oujda,580665.95,129,324,4501.286434,4.54,46


In [42]:
customer_sales = delivered_sales.merge(
    delivered_orders[["order_id", "customer_id"]],
    on="order_id",
    how="left",
    validate="many_to_one",
)

customer_sales = customer_sales.merge(
    customers[["customer_id", "customer_segment", "customer_age"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
)

customer_sales.head()

,order_item_id,order_id,product_id,quantity,unit_price_mad,discount_rate,line_total_mad,order_date,shipping_city,order_month,customer_id,customer_segment,customer_age
0,I000001,O00001,P099,3,243.24,0.10,656.75,2024-04-09,tangier,2024-04,C0713,regular,28.0
1,I000002,O00001,P014,1,3643.31,0.00,3643.31,2024-04-09,tangier,2024-04,C0713,regular,28.0
2,I000003,O00001,P071,2,2497.82,0.05,4745.86,2024-04-09,tangier,2024-04,C0713,regular,28.0
3,I000004,O00002,P075,1,2409.28,0.05,2288.82,2025-11-19,marrakech,2025-11,C0942,regular,21.0
4,I000005,O00002,P059,3,376.57,0.05,1073.22,2025-11-19,marrakech,2025-11,C0942,regular,21.0


In [44]:
segment_performance = (
    customer_sales
    .groupby("customer_segment")
    .agg(
        revenue_mad=("line_total_mad", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique"),
        units_sold=("quantity", "sum"),
    )
    .reset_index()
)

segment_performance["average_order_value_mad"] = (
    segment_performance["revenue_mad"]
    / segment_performance["orders"]
)

segment_performance["revenue_share_pct"] = (
    segment_performance["revenue_mad"]
    / segment_performance["revenue_mad"].sum()
    * 100
).round(2)

segment_performance.sort_values(
    "revenue_mad",
    ascending=False,
)

,customer_segment,revenue_mad,orders,customers,units_sold,average_order_value_mad,revenue_share_pct
2,regular,7958158.70,1631,590,4319,4879.312508,62.25
0,new,2494091.59,515,182,1409,4842.896291,19.51
1,premium,2332060.41,481,165,1312,4848.358441,18.24


In [46]:
customer_performance = (
    customer_sales
    .groupby(["customer_id", "customer_segment", ])
    .agg(
        total_revenue_mad=("line_total_mad", "sum"),
        total_orders=("order_id", "nunique"),
        total_units=("quantity", "sum"),
    )
    .reset_index()
)

customer_performance["average_order_value_mad"] = (
    customer_performance["total_revenue_mad"]
    / customer_performance["total_orders"]
)

customer_performance = customer_performance.sort_values(
    "total_revenue_mad",
    ascending=False,
)

customer_performance.head(10)

,customer_id,customer_segment,total_revenue_mad,total_orders,total_units,average_order_value_mad
293,C0307,regular,74586.92,9,35,8287.435556
783,C0837,regular,56182.77,5,24,11236.554000
432,C0457,regular,55570.17,6,22,9261.695000
673,C0715,regular,51811.29,7,24,7401.612857
225,C0235,premium,50428.85,6,21,8404.808333
770,C0823,new,49258.14,3,14,16419.380000
740,C0788,regular,48492.59,7,25,6927.512857
567,C0607,regular,47937.14,4,16,11984.285000
578,C0618,premium,47000.26,8,23,5875.032500
277,C0290,regular,46817.88,2,14,23408.940000


In [47]:
total_customer_revenue = customer_performance[
    "total_revenue_mad"
].sum()

top_10_customer_revenue = customer_performance.head(10)[
    "total_revenue_mad"
].sum()

top_10_revenue_share_pct = (
    top_10_customer_revenue
    / total_customer_revenue
    * 100
)

print(f"Top 10 customers' revenue share: {top_10_revenue_share_pct:.2f}%")

Top 10 customers' revenue share: 4.13%
